In [1]:
import geopandas as gpd
import os
import pandas as pd
import osmnx as ox
import folium
from shapely.geometry import box, Point, Polygon
from shapely.validation import make_valid
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from math import sqrt
import yaml

In [2]:
# get path to project root directory
project_root = Path.cwd().parents[0]

# build path to yaml config file
config_path = project_root / "configs"/"paths.yaml"

In [3]:
# load yaml file into python dictionary
with open(config_path) as f:
    paths = yaml.safe_load(f)

paths

{'data': {'external': 'data/external',
  'raw': 'data/raw',
  'processed': 'data/processed',
  'results': 'results',
  'imagery': 'data/imagery'},
 'outputs': {'dynamic_maps': 'outputs/maps/dynamic',
  'static_maps': 'output/maps/static',
  'figures': 'output/figures',
  'tables': 'outputs/tables'},
 'imagery': {'nakivale_sample': 'data/imagery/nakivale_sample'}}

In [4]:
# build data directory using paths from yaml config
data_dir = project_root/paths['data']['processed']

# build path to external data using paths from yaml config
data_external = project_root/paths['data']['external']

# #  build path to our deduped settlements geojson file
settlements_path = data_dir / "UNHCR_poc_boundaries-Uganda_attributed_deduped.geojson"

# build path to our regions geojson file
refugee_regions_path = data_external / "refugeehosting_regions.geojson"

In [5]:
# build path to output directory for osm road data
output_dir = project_root/paths['data']['processed']

# build path to output directory for maps
maps_dir = project_root/paths['outputs']['dynamic_maps']

In [6]:
refugee_regions = gpd.read_file(refugee_regions_path)
refugee_regions = gpd.GeoDataFrame(refugee_regions[['ADM2_EN','ADM1_EN', 'ADM0_EN','geometry']], geometry = 'geometry')
refugee_regions["geometry"] = refugee_regions.geometry.apply(make_valid)
refugee_regions = refugee_regions.to_crs(4326)
refugee_regions

,ADM2_EN,ADM1_EN,ADM0_EN,geometry
0,Adjumani,Northern,Uganda,"MULTIPOLYGON (((32.04597 3.58896, 32.04578 3.5..."
1,Amuru,Northern,Uganda,"MULTIPOLYGON (((32.06289 3.57969, 32.06281 3.5..."
2,Arua,Northern,Uganda,"MULTIPOLYGON (((31.17434 3.3474, 31.17463 3.34..."
3,Isingiro,Western,Uganda,"MULTIPOLYGON (((30.77797 -0.60322, 30.77799 -0..."
4,Kamwenge,Western,Uganda,"MULTIPOLYGON (((30.46881 0.57351, 30.46875 0.5..."
5,Kikuube,Western,Uganda,"MULTIPOLYGON (((30.89045 1.54694, 30.91887 1.5..."
6,Kiruhura,Western,Uganda,"MULTIPOLYGON (((31.05885 0.0435, 31.05845 0.04..."
7,Kiryandongo,Western,Uganda,"MULTIPOLYGON (((31.83729 2.36609, 31.83792 2.3..."
8,Kyegegwa,Western,Uganda,"MULTIPOLYGON (((30.91451 0.75863, 30.91575 0.7..."
9,Lamwo,Northern,Uganda,"MULTIPOLYGON (((33.16747 3.78522, 33.17364 3.7..."


In [7]:
from sqlalchemy import create_engine
import psycopg2
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

DB = os.getenv("PGDATABASE")
USER = os.getenv("PGUSER")
PWD = os.getenv("PGPASSWORD")
HOST = os.getenv("PGHOST")
PORT = os.getenv("PGPORT")


conn = psycopg2.connect(
    dbname=DB,
    user=USER,
    password=PWD,
    host=HOST,
    port=PORT
)

engine = create_engine(f"postgresql://{USER}:{PWD}@{HOST}:{PORT}/{DB}")

In [8]:
target_crs = "EPSG:32636"

# build paths to our hexbins
areas = [250, 62, 15]

grid_paths = {
    f"grid_{area}k": data_dir/f"uganda_grids_{area}k_lcluc_v1.geojson" for area in areas
}
grid_250k_path = grid_paths['grid_250k']
grid_62k_path = grid_paths['grid_62k']
grid_15k_path = grid_paths['grid_15k']

# build paths to other datasets
roads_path = data_dir/"uganda_refugee_regions_osm_roads_clean_v1.geojson"
markets_path = data_dir/"uganda_markets_clean_v1.geojson"

In [9]:
# grid_250k = gpd.read_file(grid_250k_path).to_crs(32636)
# grid_62k = gpd.read_file(grid_62k_path).to_crs(32636)
# grid_15k = gpd.read_file(grid_15k_path).to_crs(32636)
# roads = gpd.read_file(roads_path).to_crs(32636)
# markets = gpd.read_file(markets_path).to_crs(32636)

In [10]:
import geoalchemy2

# grid_250k.to_postgis("grid_250k", engine, if_exists="replace", index=False)
# grid_62k.to_postgis("grid_62k", engine, if_exists="replace", index=False)
# grid_15k.to_postgis("grid_15k", engine, if_exists="replace", index=False)

In [11]:
def run_sql(sql):
    cur = conn.cursor()
    try:
        cur.execute(sql)
        conn.commit()
    except Exception as e:
        conn.rollback()
        print("SQL error:", e)
    cur.close()

In [12]:
areas = [250, 62, 15]

for area in areas:
    query = f"""
            ALTER TABLE grid_{area}k 
                ADD COLUMN IF NOT EXISTS centroid geometry(Point, 32636);
            UPDATE grid_{area}k 
                SET centroid = ST_Centroid(geometry)
            WHERE centroid IS NULL;
            """
    run_sql(query)

In [13]:
run_sql("""
CREATE INDEX IF NOT EXISTS roads_gist 
    ON roads USING GIST (geometry);
""")

run_sql("""
CREATE INDEX IF NOT EXISTS markets_gist 
    ON markets USING GIST (geometry);
""")

run_sql("""
CREATE INDEX IF NOT EXISTS grid_250k_centroid_gist 
    ON grid_250k USING GIST (centroid);
""")

run_sql("""
CREATE INDEX IF NOT EXISTS grid_62k_centroid_gist 
    ON grid_62k USING GIST (centroid);
""")

run_sql("""
CREATE INDEX IF NOT EXISTS grid_15k_centroid_gist 
    ON grid_15k USING GIST (centroid);
""")

In [14]:
for area in areas:
    query = f"""
            ALTER TABLE grid_{area}k 
                ADD COLUMN IF NOT EXISTS dist_to_road double precision;

            UPDATE grid_{area}k h
            SET dist_to_road =
            (
                SELECT ST_Distance(h.centroid, r.geometry)
                FROM roads r
                ORDER BY h.centroid <-> r.geometry
                LIMIT 1
            )
            WHERE dist_to_road IS NULL;
            """
    run_sql(query)

In [15]:
for area in areas:
    query = f"""
            ALTER TABLE grid_{area}k 
                ADD COLUMN IF NOT EXISTS dist_to_market double precision;

            UPDATE grid_{area}k h
            SET dist_to_market =
            (
                SELECT ST_Distance(h.centroid, m.geometry)
                FROM markets m
                ORDER BY h.centroid <-> m.geometry
                LIMIT 1
            )
            WHERE dist_to_market IS NULL;
            """
    run_sql(query)

In [16]:
rivers = ["rivers", "rivers_and_streams","rivers_plus"]

for area in areas:
    for river in rivers:
        query = f"""
                ALTER TABLE grid_{area}k 
                    ADD COLUMN IF NOT EXISTS dist_to_{river} double precision;

                UPDATE grid_{area}k h
                SET dist_to_{river} =
                (
                    SELECT ST_Distance(h.centroid, r.geometry)
                    FROM {river} r
                    ORDER BY h.centroid <-> r.geometry
                    LIMIT 1
                )
                WHERE dist_to_{river} IS NULL;
                """
        run_sql(query)

In [17]:
pop_centers = ['pop_centers_1', 'pop_centers_2', 'pop_centers_3']

for area in areas:
    for center in pop_centers:
        suffix = center.split("_")[-1]
        colname = f"dist_to_pop_center_{suffix}"
        query = f"""
                ALTER TABLE grid_{area}k 
                    ADD COLUMN IF NOT EXISTS {colname} double precision;

                UPDATE grid_{area}k h
                SET {colname} =
                (
                    SELECT ST_Distance(h.centroid, p.geometry)
                    FROM {center} p
                    ORDER BY h.centroid <-> p.geometry
                    LIMIT 1
                )
                WHERE dist_to_pop_center_{suffix} IS NULL;
                """
        run_sql(query)

In [18]:
grid_250k_processed = gpd.read_postgis(
    "SELECT * FROM grid_250k", 
    con=engine, 
    geom_col="geometry"
)

grid_62k_processed = gpd.read_postgis(
    "SELECT * FROM grid_62k", 
    con=engine, 
    geom_col="geometry"
)

grid_15k_processed = gpd.read_postgis(
    "SELECT * FROM grid_15k", 
    con=engine, 
    geom_col="geometry"
)

In [19]:
grid_250k_processed.sort_values(by = 'settlement_name').head(3)

,OID,ADM2_EN,ADM1_EN,ADM0_EN,settlement_name,lat,lon,geometry,centroid,dist_to_road,dist_to_market,dist_to_rivers,dist_to_rivers_plus,dist_to_rivers_and_streams,dist_to_pop_center_1,dist_to_pop_center_2,dist_to_pop_center_3
92030,91896,Adjumani,Northern,Uganda,Agojo,3.393625,31.745558,"POLYGON ((360894.038 374942.248, 360894.038 37...",01010000207C7F0000FA1E2A2710031641892E3AFE60E6...,334.605946,2099.899679,9667.851543,2344.531402,2344.531402,584.084324,584.084324,5332.587147
91129,90995,Adjumani,Northern,Uganda,Agojo,3.402646,31.727546,"POLYGON ((358894.038 375942.248, 358894.038 37...",01010000207C7F0000F61E2A27D0E315418B2E3AFE00F6...,90.045227,1004.724928,7629.408811,323.380033,323.380033,1697.677162,1697.677162,7559.257865
90933,90799,Adjumani,Northern,Uganda,Agojo,3.434297,31.723004,"POLYGON ((358394.038 379442.248, 358394.038 37...",01010000207C7F0000F81E2A2700DC15418C2E3AFEB02C...,224.189215,3080.739777,5239.951582,3016.294288,3016.294288,415.583277,415.583277,9828.408091


In [ ]:
# grid_250k_processed.to_file(output_dir/'uganda_grids_250k_lcluc_processed_v1.geojson', driver = 'GeoJSON')
# grid_62k_processed.to_file(output_dir/'uganda_grids_62k_lcluc_processed_v1.geojson', driver = 'GeoJSON')
# grid_15k_processed.to_file(output_dir/'uganda_grids_15k_lcluc_processed_v1.geojson', driver = 'GeoJSON')

### Geodesic Distances and Comparison

In [21]:
for area in areas:
    query = f"""CREATE TABLE grid_{area}k_v2 (LIKE grid_{area}k INCLUDING ALL);
                INSERT INTO grid_{area}k_v2
                SELECT *
                FROM grid_{area}k;
            """
    run_sql(query)

SQL error: relation "grid_250k_v2" already exists

SQL error: relation "grid_62k_v2" already exists

SQL error: relation "grid_15k_v2" already exists



In [22]:
run_sql("""
CREATE INDEX IF NOT EXISTS grid_250k_v2_centroid_gist 
    ON grid_250k_v2 USING GIST (centroid);
""")

run_sql("""
CREATE INDEX IF NOT EXISTS grid_62k_v2_centroid_gist 
    ON grid_62k_v2 USING GIST (centroid);
""")

run_sql("""
CREATE INDEX IF NOT EXISTS grid_15k_v2_centroid_gist 
    ON grid_15k_v2 USING GIST (centroid);
""")

In [23]:
for area in areas:
    query = f"""
        -- Add column if missing
        ALTER TABLE grid_{area}k_v2
            ADD COLUMN IF NOT EXISTS dist_to_road_geodesic double precision;

        -- Update using geodesic distance (earth surface)
        UPDATE grid_{area}k_v2 AS h
        SET dist_to_road_geodesic =
        (
            SELECT ST_DistanceSphere(
                ST_Transform(h.centroid, 4326),
                ST_Transform(r.geometry, 4326)
            )
            FROM roads AS r
            ORDER BY h.centroid <-> r.geometry   -- KNN search (projected CRS)
            LIMIT 1
        )
        WHERE dist_to_road_geodesic IS NULL;
    """
    run_sql(query)

In [24]:
for area in areas:
    query = f"""
            ALTER TABLE grid_{area}k_v2 
                ADD COLUMN IF NOT EXISTS dist_to_market_geodesic double precision;

            UPDATE grid_{area}k_v2 AS h
            SET dist_to_market_geodesic =
            (
                SELECT ST_DistanceSphere(
                    ST_Transform(h.centroid, 4326),
                    ST_Transform(m.geometry, 4326)
                )
                FROM markets m
                ORDER BY h.centroid <-> m.geometry
                LIMIT 1
            )
            WHERE dist_to_market_geodesic IS NULL;
            """
    run_sql(query)

In [25]:
rivers = ["rivers", "rivers_and_streams", "rivers_plus"]

for area in areas:
    for river in rivers:

        colname = f"dist_to_{river}_geodesic"  # Keep your naming convention

        query = f"""
            -- Add column
            ALTER TABLE grid_{area}k_v2 
                ADD COLUMN IF NOT EXISTS {colname} double precision;

            -- Compute geodesic distance to nearest feature in {river}
            UPDATE grid_{area}k_v2 AS h
            SET {colname} =
            (
                SELECT ST_DistanceSphere(
                    ST_Transform(h.centroid, 4326),
                    ST_Transform(r.geometry, 4326)
                )
                FROM {river} AS r
                ORDER BY h.centroid <-> r.geometry      -- projected-space KNN
                LIMIT 1
            )
            WHERE {colname} IS NULL;
        """

        run_sql(query)

In [26]:
pop_centers = ['pop_centers_1', 'pop_centers_2', 'pop_centers_3']
areas = [250, 62, 15]

for area in areas:
    for center in pop_centers:

        suffix = center.split("_")[-1]
        colname = f"dist_to_pop_center_{suffix}_geodesic"

        query = f"""
            -- Add column
            ALTER TABLE grid_{area}k_v2 
                ADD COLUMN IF NOT EXISTS {colname} double precision;

            -- Compute geodesic (earth surface) distance
            UPDATE grid_{area}k_v2 AS h
            SET {colname} =
            (
                SELECT 
                    ST_DistanceSphere(
                        ST_Transform(h.centroid, 4326),
                        ST_Transform(p.geometry, 4326)
                    )
                FROM {center} AS p
                ORDER BY h.centroid <-> p.geometry
                LIMIT 1
            )
            WHERE {colname} IS NULL;
        """

        run_sql(query)

In [ ]:
distance_pairs = [
    ("dist_to_roads", "dist_to_roads_geodesic"),
    ("dist_to_markets", "dist_to_markets_geodesic"),
    ("dist_to_rivers", "dist_to_rivers_geodesic"),
    ("dist_to_rivers_and_streams", "dist_to_rivers_and_streams_geodesic"),
    ("dist_to_rivers_plus", "dist_to_rivers_plus_geodesic"),
    ("dist_to_pop_center_1", "dist_to_pop_center_1_geodesic"),
    ("dist_to_pop_center_2", "dist_to_pop_center_2_geodesic"),
    ("dist_to_pop_center_3", "dist_to_pop_center_3_geodesic"),
]

for area in areas:
    for planar, geo in distance_pairs:

        diff_col = f"{planar}_diff"

        query = f"""
            -- Add difference column
            ALTER TABLE grid_{area}k_v2 
                ADD COLUMN IF NOT EXISTS {diff_col} double precision;

            -- Compute absolute difference
            UPDATE grid_{area}k_v2
            SET grid_{area}k_v2.{diff_col} = ABS(grid_{area}k_v2.{planar} - grid_{area}k_v2.{geo})
            WHERE grid_{area}k_v2.{diff_col} IS NULL
              AND grid_{area}k_v2.{planar} IS NOT NULL
              AND grid_{area}k_v2.{geo} IS NOT NULL;
        """

        run_sql(query)


SQL error: column "dist_to_roads" does not exist
LINE 10:               AND dist_to_roads IS NOT NULL
                           ^
HINT:  Perhaps you meant to reference the column "grid_250k_v2.dist_to_road".

SQL error: column "dist_to_markets" does not exist
LINE 10:               AND dist_to_markets IS NOT NULL
                           ^
HINT:  Perhaps you meant to reference the column "grid_250k_v2.dist_to_market".

SQL error: column "dist_to_roads" does not exist
LINE 10:               AND dist_to_roads IS NOT NULL
                           ^
HINT:  Perhaps you meant to reference the column "grid_62k_v2.dist_to_road".

SQL error: column "dist_to_markets" does not exist
LINE 10:               AND dist_to_markets IS NOT NULL
                           ^
HINT:  Perhaps you meant to reference the column "grid_62k_v2.dist_to_market".

SQL error: column "dist_to_roads" does not exist
LINE 10:               AND dist_to_roads IS NOT NULL
                           ^
HINT:  Perhaps you 